# Lab 2 - Tsunami Across the Ocean

A long wave moves from deep water toward a shallow coast. The sponge layer reduces reflections at the far boundary.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from shallowwater import (ModelParams, make_grid, setup_initial_state, compute_dt_cfl, run_model,
                          zero_forcing, shelf_bathymetry, wave_speed, make_sponge_hook)


In [ ]:
Nx, Ny = 180, 48
Lx, Ly = 3.0e6, 8.0e5
grid = make_grid(Nx, Ny, Lx, Ly)
H = shelf_bathymetry(grid, H_deep=4000.0, H_coast=100.0, shelf_width=900e3, coast='east')
params = ModelParams(H=H, g=9.81, f0=0.0, beta=0.0, r=0.0, linear=True)
dt = compute_dt_cfl(grid, params, cfl=0.45)
plt.plot(grid.x_c/1000, H[Ny//2])
plt.gca().invert_yaxis()
plt.xlabel('x [km]')
plt.ylabel('depth H [m]')
plt.title('Deep ocean to shallow coast')


In [ ]:
def ic_tsunami(g, p):
    return setup_initial_state(g, p, mode='gaussian_bump', amp=0.25, x0=0.25*g.Lx, y0=0.5*g.Ly, R=1.2e5)

sponge = make_sponge_hook(width=350e3, tau_min=900.0, sides=('west',))
out = run_model(tmax=4.5*3600, dt=dt, grid=grid, params=params, forcing_fn=zero_forcing,
                ic_fn=ic_tsunami, save_every=12, out_vars=('eta',), hooks=[sponge])
coastal_eta = out['eta'][:, :, -4].max(axis=1)
plt.plot(out['t']/3600, coastal_eta)
plt.xlabel('time [hours]')
plt.ylabel('near-coast max eta [m]')
plt.title('When does the largest coastal wave arrive?')


Repeat with `H_coast=50` and `H_coast=300`. Which coast is more dangerous in this model?